In [ ]:
# ensemble_train_project_full.py
import os
import random
from pathlib import Path
from typing import List

import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

import timm
from sklearn.metrics import roc_auc_score

# -------------------------
# Config (edit as needed)
# -------------------------
# Set to the folder that contains the 'Train' directory
DATA_DIR = r"C:\Users\rajam\Desktop\Project"   # parent folder containing 'Train'
TRAIN_SUBFOLDER = "Train"                      # matches the folder name inside DATA_DIR
LOCAL_WEIGHTS_DIR = r"C:\Users\rajam\Desktop\Project\checkpoints"  # folder where you placed xception-*.pth etc.

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BACKBONES = [
    ("xception", True, 299),
    ("efficientnet_b4", True, 380),
    ("swin_base_patch4_window7_224", True, 224),
]

NUM_CLASSES = 1
EPOCHS = 1   # single epoch for quick test
LR = 1e-4
WEIGHT_DECAY = 1e-5
SAVE_DIR = os.path.join(DATA_DIR, "trained_checkpoints")
os.makedirs(SAVE_DIR, exist_ok=True)


# -------------------------
# Robust dataset loader (case-insensitive Real/Fake support)
# -------------------------
class ImageFolderBinary(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root = Path(root_dir)
        self.transform = transform
        self.samples = []

        def find_subfolder(base: Path, name: str):
            cand = base / name
            if cand.exists() and cand.is_dir():
                return cand
            for p in base.iterdir():
                if p.is_dir() and p.name.lower() == name.lower():
                    return p
            return None

        real_folder = find_subfolder(self.root, "real")
        fake_folder = find_subfolder(self.root, "fake")

        if real_folder and fake_folder:
            for p in real_folder.iterdir():
                if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]:
                    self.samples.append((str(p), 0))
            for p in fake_folder.iterdir():
                if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]:
                    self.samples.append((str(p), 1))
        else:
            # fallback: try inferring labels from filenames
            for p in self.root.rglob("*"):
                if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]:
                    name = p.name.lower()
                    if "real" in name:
                        self.samples.append((str(p), 0))
                    elif "fake" in name:
                        self.samples.append((str(p), 1))

        if len(self.samples) == 0:
            raise ValueError(f"No images found in {root_dir}. Expected real/ and fake/ subfolders (any case), or filenames containing 'real'/'fake'.")
        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, float(label)


# -------------------------
# Model wrappers (loads local weights if present)
# -------------------------
class SingleBackbone(nn.Module):
    def __init__(self, model_name: str, pretrained: bool, out_dim=1,
                 local_weights_dir: str = None):
        """
        model_name: timm model string (e.g. 'xception')
        pretrained: if True -> try to use local checkpoint first, otherwise allow timm to download
        local_weights_dir: optional folder where you keep pretrained files (e.g. r"C:\...Project\checkpoints")
        """
        super().__init__()

        # create backbone WITHOUT classifier head (we use features)
        self.model = timm.create_model(model_name, pretrained=False, num_classes=0, global_pool="avg")
        feat_dim = self.model.num_features

        # optionally load local pretrained weights if provided and available
        loaded_local = False
        if pretrained and local_weights_dir is not None:
            try:
                local_dir = Path(local_weights_dir)
                if local_dir.exists():
                    # try several matching patterns
                    possible_files = list(local_dir.glob(f"{model_name}*.pth")) + list(local_dir.glob(f"*{model_name}*.pth"))
                    if possible_files:
                        ckpt_path = str(possible_files[0])
                        print(f"Loading local pretrained backbone weights from: {ckpt_path}")
                        sd = torch.load(ckpt_path, map_location="cpu")
                        missing, unexpected = self.model.load_state_dict(sd, strict=False)
                        if missing:
                            print("Note: missing keys when loading local weights (expected for num_classes=0):", len(missing))
                        if unexpected:
                            print("Note: unexpected keys when loading local weights:", len(unexpected))
                        loaded_local = True
            except Exception as e:
                print("Failed to load local checkpoint (continuing):", e)

        # If pretrained True but no local weights loaded, let timm download official pretrained weights (fallback)
        if pretrained and not loaded_local:
            try:
                print(f"Attempting to load timm pretrained weights for '{model_name}' (may download if not cached).")
                ref_model = timm.create_model(model_name, pretrained=True)  # this may download
                ref_sd = ref_model.state_dict()
                missing, unexpected = self.model.load_state_dict(ref_sd, strict=False)
                print(f"Loaded timm pretrained weights for {model_name}.")
            except Exception as e:
                print("Could not auto-load timm pretrained weights:", e)
                print("Continuing with randomly initialized backbone.")

        # build classification head on top of the pooled features
        self.head = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(feat_dim // 2, out_dim)
        )

    def forward(self, x):
        f = self.model(x)   # pooled features (B, feat_dim)
        logits = self.head(f).squeeze(1)
        return logits


class WeightedEnsemble(nn.Module):
    def __init__(self, models: List[nn.Module], learnable: bool = False):
        super().__init__()
        self.models = nn.ModuleList(models)
        self.n = len(models)
        if learnable:
            self.weight_logits = nn.Parameter(torch.zeros(self.n))
        else:
            self.register_buffer("weight_logits", torch.zeros(self.n))

    def forward(self, x):
        probs = []
        for m in self.models:
            logits = m(x)
            p = torch.sigmoid(logits)
            probs.append(p)
        probs = torch.stack(probs, dim=1)
        wl = self.weight_logits.to(probs.device)
        weights = F.softmax(wl, dim=0)
        weighted = (probs * weights.unsqueeze(0)).sum(dim=1)
        return weighted

    def set_weights(self, weights_list: List[float]):
        w = torch.tensor(weights_list, dtype=torch.float32)
        if isinstance(self.weight_logits, nn.Parameter):
            with torch.no_grad():
                self.weight_logits.copy_(w)
        else:
            self.weight_logits.copy_(w)


# -------------------------
# Utils (transforms, eval, auc)
# -------------------------
def get_transforms(size):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([transforms.ColorJitter(0.2, 0.2, 0.2, 0.02)], p=0.5),
        transforms.RandomAdjustSharpness(0.2, p=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])


def eval_model_probs(model, loader, device):
    model.eval()
    preds = []
    labels = []
    with torch.no_grad():
        total = len(loader)
        progress = tqdm(enumerate(loader), total=total, ncols=100, desc="Eval")
        for batch_idx, (imgs, labs) in progress:
            imgs = imgs.to(device)
            out = model(imgs)
            if out.max() > 1.01 or out.min() < -0.01:
                out = torch.sigmoid(out)
            preds.extend(out.detach().cpu().numpy().tolist())
            labels.extend(labs)
            percent = (batch_idx + 1) / total * 100
            progress.set_postfix({"Progress": f"{percent:.1f}%"})
    return np.array(preds), np.array(labels)


def compute_auc(labels, preds):
    try:
        return roc_auc_score(labels, preds)
    except Exception:
        return float("nan")


def derive_weights_from_val(models, val_loader, device):
    aucs = []
    for idx, m in enumerate(models):
        print(f"Computing val AUC for model {idx}")
        p, l = eval_model_probs(m.to(device), val_loader, device)
        auc = compute_auc(l, p)
        print(f"Model {idx} AUC = {auc:.4f}")
        aucs.append(max(auc, 0.5))
    aucs = np.array(aucs)
    exps = np.exp(aucs - aucs.max())
    weights = exps / exps.sum()
    print("Derived normalized weights:", weights)
    return weights.tolist()


# -------------------------
# Training routines (with progress % bars)
# -------------------------
def train_singlebackbone(model, train_loader, val_loader, device, epochs=EPOCHS, lr=LR, model_name="model"):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    bce = nn.BCEWithLogitsLoss()
    best_val_auc = 0.0
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        total_batches = len(train_loader)
        progress = tqdm(enumerate(train_loader), total=total_batches, ncols=120,
                        desc=f"Training {model_name} | Epoch {epoch+1}/{epochs}")
        for batch_idx, (imgs, labs) in progress:
            imgs = imgs.to(device)
            labels = labs.to(device)
            logits = model(imgs)
            loss = bce(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            percent = (batch_idx + 1) / total_batches * 100
            progress.set_postfix({"Loss": f"{loss.item():.4f}", "Progress": f"{percent:.1f}%"})
        scheduler.step()
        preds, labs = eval_model_probs(model, val_loader, device)
        val_auc = compute_auc(labs, preds)
        avg_loss = running_loss / (len(train_loader.dataset) + 1e-12)
        print(f"{model_name} Epoch {epoch+1}/{epochs} avg_loss={avg_loss:.4f} val_auc={val_auc:.4f}")
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{model_name}_best.pth"))
    print(f"Best val AUC for {model_name}: {best_val_auc:.4f}")
    return best_val_auc


def train_ensemble_with_trainable_weights(models: List[nn.Module], train_loader, val_loader, device, epochs=5):
    for m in models:
        for p in m.parameters():
            p.requires_grad = False
    ensemble = WeightedEnsemble(models=models, learnable=True).to(device)
    opt = torch.optim.AdamW([p for p in ensemble.parameters() if p.requires_grad], lr=1e-3)
    bce = nn.BCELoss()
    best_auc = 0.0
    for epoch in range(epochs):
        ensemble.train()
        running_loss = 0.0
        total_batches = len(train_loader)
        progress = tqdm(enumerate(train_loader), total=total_batches, ncols=120,
                        desc=f"Training Ensemble Weights | Epoch {epoch+1}/{epochs}")
        for batch_idx, (imgs, labs) in progress:
            imgs = imgs.to(device)
            labs = labs.to(device)
            probs = ensemble(imgs)
            loss = bce(probs, labs)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running_loss += loss.item() * imgs.size(0)
            percent = (batch_idx + 1) / total_batches * 100
            progress.set_postfix({"Loss": f"{loss.item():.4f}", "Progress": f"{percent:.1f}%"})
        preds, labels = eval_model_probs(ensemble, val_loader, device)
        val_auc = compute_auc(labels, preds)
        avg_loss = running_loss / (len(train_loader.dataset) + 1e-12)
        print(f"Ensemble weights Epoch {epoch+1}/{epochs} avg_loss={avg_loss:.4f} val_auc={val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(ensemble.state_dict(), os.path.join(SAVE_DIR, "ensemble_trainable_best.pth"))
    print(f"Best ensemble val AUC: {best_auc:.4f}")
    return ensemble


# -------------------------
# Main: use only train folder (train/Real & train/Fake)
# -------------------------
def main():
    sizes = [s for (_, _, s) in BACKBONES]
    max_size = max(sizes)
    transform_train = get_transforms(max_size)
    transform_val = transforms.Compose([
        transforms.Resize((max_size, max_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    # build train_root path (pointing to PROJECT\Train)
    # If DATA_DIR already points at Train, use it directly
    base = Path(DATA_DIR)
    candidate = base / TRAIN_SUBFOLDER
    if candidate.exists():
        train_root = str(candidate)
    elif base.exists() and (base / "Real").exists() and (base / "Fake").exists():
        train_root = str(base)
    else:
        # try to find Train folder under DATA_DIR or user's home
        found = None
        for p in base.rglob(TRAIN_SUBFOLDER):
            if p.is_dir():
                found = p
                break
        if found is not None:
            train_root = str(found)
        else:
            raise FileNotFoundError(f"Could not locate '{TRAIN_SUBFOLDER}' folder under {DATA_DIR}. Set DATA_DIR correctly.")

    print("Loading train folder:", train_root)
    train_ds = ImageFolderBinary(train_root, transform=transform_train)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

    # val/test use train_loader as dummy (user requested training only)
    val_loader = train_loader
    test_loader = train_loader

    models = []
    for idx, (name, pretrained, img_size) in enumerate(BACKBONES):
        print(f"Initializing backbone {name} (pretrained={pretrained})")
        # use local weights dir so SingleBackbone will attempt to load local checkpoint first
        m = SingleBackbone(model_name=name, pretrained=pretrained, out_dim=1, local_weights_dir=LOCAL_WEIGHTS_DIR)
        model_name = f"model_{idx}_{name}"
        try:
            train_singlebackbone(m, train_loader, val_loader, DEVICE, epochs=EPOCHS, lr=LR, model_name=model_name)
        except Exception as e:
            print("Warning: training failed for", model_name, "->", e)
        best_path = os.path.join(SAVE_DIR, f"{model_name}_best.pth")
        if os.path.exists(best_path):
            m.load_state_dict(torch.load(best_path, map_location=DEVICE))
        models.append(m)

    print("Deriving static weights from validation AUCs (val == train here)...")
    weights = derive_weights_from_val(models, val_loader, DEVICE)
    ensemble_static = WeightedEnsemble(models=models, learnable=False)
    ensemble_static.set_weights(weights)
    ensemble_static = ensemble_static.to(DEVICE)
    preds, labs = eval_model_probs(ensemble_static, test_loader, DEVICE)
    auc = compute_auc(labs, preds)
    print(f"Static-weight ensemble test AUC = {auc:.4f}")

    print("Training trainable-weight ensemble (weights only)...")
    ensemble_trainable = train_ensemble_with_trainable_weights(models, train_loader, val_loader, DEVICE, epochs=5)
    preds2, labs2 = eval_model_probs(ensemble_trainable, test_loader, DEVICE)
    auc2 = compute_auc(labs2, preds2)
    print(f"Trainable-weight ensemble test AUC = {auc2:.4f}")

    torch.save(ensemble_static.state_dict(), os.path.join(SAVE_DIR, "ensemble_static.pth"))
    torch.save(ensemble_trainable.state_dict(), os.path.join(SAVE_DIR, "ensemble_trainable.pth"))
    print("Done. Check", SAVE_DIR)


if __name__ == "__main__":
    main()


<>:109: SyntaxWarning: invalid escape sequence '\.'
<>:109: SyntaxWarning: invalid escape sequence '\.'
C:\Users\rajam\AppData\Local\Temp\ipykernel_27680\228390415.py:109: SyntaxWarning: invalid escape sequence '\.'
  local_weights_dir: optional folder where you keep pretrained files (e.g. r"C:\...Project\checkpoints")
c:\Users\rajam\Desktop\Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading train folder: C:\Users\rajam\Desktop\Project\Train
Initializing backbone xception (pretrained=True)


c:\Users\rajam\Desktop\Project\venv\Lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Loading local pretrained backbone weights from: C:\Users\rajam\Desktop\Project\checkpoints\xception-43020ad28.pth
Note: unexpected keys when loading local weights: 2
